# Section 5.6 — Sensitivity to Stages

Reproduces Table 8 from the thesis.  
Runs **EEV, SP, WS** on a **60-cage** synthetic fleet across 2-stage (9 sc.), 3-stage (27 sc.), and 4-stage (81 sc.) scenario trees. DE is not attempted (OOM at 60 cages).

| Config | Stages | Scenarios | Stage slices (months) | EEV fixes |
|--------|--------|-----------|----------------------|-----------|
| 2-stage | 2 | 3² = 9  | 0-29, 30-59           | months 0-29  |
| 3-stage | 3 | 3³ = 27 | 0-19, 20-39, 40-59    | months 0-39  |
| 4-stage | 4 | 3⁴ = 81 | 0-14, 15-29, 30-44, 45-59 | months 0-44 |


In [1]:
import sys
import os
import importlib.util

_here   = os.path.dirname(os.path.abspath('__file__'))
_models = os.path.join(_here, '..', 'models')
sys.path.insert(0, _models)
sys.path.insert(0, _here)   # 5.6/instance.py takes priority over models/instance.py

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df,
    temps_bad_12, temps_normal_12, temps_good_12,
    S_normal_v, S_bad_v,
)
from IP import SalmonFarmingMILP

# Load SP classes — 2SP.py / 3SP.py start with digits so use importlib
def _load_ald(filename):
    spec = importlib.util.spec_from_file_location('_sp', os.path.join(_here, filename))
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.AugmentedLagrangianDecomposition

ALD2 = _load_ald('2SP.py')
ALD3 = _load_ald('3SP.py')

# 4-stage SP lives in models/
from SP import AugmentedLagrangianDecomposition as ALD4

print('Imports OK')


Imports OK


In [2]:
# 60-cage fleet — use instance.py directly (Loc1-Loc6, real MAB values)

units_df_60 = units_df.copy().reset_index(drop=True)
loc_mab_60  = loc_mab

print(f'Fleet: {len(units_df_60)} cages, {units_df_60["location"].nunique()} locations')
print(units_df_60.groupby('location').size().to_frame('cages'))
print(f'Regional MAB: {regional_mab:,}')


Fleet: 60 cages, 6 locations
          cages
location       
Loc1         10
Loc2         12
Loc3          8
Loc4         10
Loc5          8
Loc6         12
Regional MAB: 35,000,000.0


In [3]:
# Stage configurations

TEMP_MAP = {
    'bad':    temps_bad_12,
    'normal': temps_normal_12,
    'good':   temps_good_12,
}
LABELS = ['bad', 'normal', 'good']

STAGE_CFG = {
    2: {
        'slices':     [list(range(0, 30)), list(range(30, T))],
        'na_months':  set(range(0, 30)),
        'ald_class':  ALD2,
    },
    3: {
        'slices':     [list(range(0, 20)), list(range(20, 40)), list(range(40, T))],
        'na_months':  set(range(0, 40)),
        'ald_class':  ALD3,
    },
    4: {
        'slices':     [list(range(0, 15)), list(range(15, 30)), list(range(30, 45)), list(range(45, T))],
        'na_months':  set(range(0, 45)),
        'ald_class':  ALD4,
    },
}


def _tile(arr, n):
    return np.tile(arr, (n // 12) + 1)[:n]


def build_scenarios_for(n_stages):
    """Build all 3^n_stages (name, temps, S, prob) tuples for the given stage count."""
    slices = STAGE_CFG[n_stages]['slices']
    n_sc   = 3 ** n_stages

    from itertools import product
    scenarios = []
    for combo in product(LABELS, repeat=n_stages):
        name     = '__'.join(f's{i+1}_{sl}' for i, sl in enumerate(combo))
        temps_sc = np.zeros(T)
        S_sc     = np.full(T, S_normal_v)
        for i, (sl, months) in enumerate(zip(combo, slices)):
            prev = combo[i - 1] if i > 0 else None
            surv = S_bad_v if (sl == 'good' and prev == 'good') else S_normal_v
            for mm in months:
                S_sc[mm] = surv
            temps_sc[months] = _tile(TEMP_MAP[sl], len(months))
        scenarios.append((name, temps_sc, S_sc, 1.0 / n_sc))
    return scenarios


In [4]:
# WS helper

def run_ws_for(units_df_sub, loc_mab_sub, n_stages, mip_gap=0.02):
    scenarios = build_scenarios_for(n_stages)
    n_feas = 0
    ws_obj = 0.0
    t0 = time.time()
    for sc_name, temps_sc, S_sc, prob in scenarios:
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab_sub, regional_mab=regional_mab,
            scenario_name=sc_name,
        )
        milp.model.Params.OutputFlag = 0
        milp.model.Params.MIPGap = mip_gap
        milp.model.optimize()
        if milp.model.SolCount > 0:
            n_feas += 1
            ws_obj += prob * milp.model.ObjVal
    return ws_obj, time.time() - t0, n_feas


# EEV helper

def run_eev_for(units_df_sub, loc_mab_sub, n_stages, mip_gap=0.02):
    na_months = STAGE_CFG[n_stages]['na_months']

    # Step 1: EV model (expected-parameter deterministic IP)
    temps_exp = np.tile(
        np.array([5, 5, 5, 6, 9, 12, 14, 16.5, 15.5, 13, 10, 7.5]),
        (T // 12) + 1
    )[:T]
    S_exp = np.full(T, (2.0 * S_normal_v + S_bad_v) / 3.0)

    t0 = time.time()
    ev_m = SalmonFarmingMILP(
        units_df=units_df_sub, temps_t=temps_exp, survival_rates=S_exp,
        horizon_months=T, loc_mab=loc_mab_sub, regional_mab=regional_mab,
        density_limit=25.0, verbose=False,
    )
    ev_m.model.Params.OutputFlag = 0
    ev_m.model.Params.MIPGap = mip_gap
    ev_m.model.optimize()
    if ev_m.model.SolCount == 0:
        print('  EV model infeasible')
        return None, time.time() - t0, 0

    # Step 2: Extract decisions for fixed stages
    ev_stk = {
        (u, t): round(ev_m.variables['z'][u, t].X)
        for u in ev_m.U for t in ev_m.Tset if t in na_months
    }
    ev_harv = {
        (u, s, t): round(ev_m.variables['h'][u, s, t].X)
        for u in ev_m.U for s in ev_m.Tset
        for t in ev_m.H_by_us.get((u, s), []) if t in na_months
    }
    ev_hexist = {
        (u, t): round(ev_m.variables['h_exist'][u, t].X)
        for u in ev_m.U_exist for t in ev_m.Tset if t in na_months
    }
    ev_q = {
        (u, s): ev_m.variables['q'][u, s].X
        for u in ev_m.U for s in ev_m.Tset
        if s in na_months and round(ev_m.variables['z'][u, s].X) == 1
    }

    # Step 3: Evaluate across all n_stages scenarios
    scenarios = build_scenarios_for(n_stages)
    n_feas = 0
    eev_obj = 0.0
    feas_prob = 0.0

    for sc_name, temps_sc, S_sc, prob in scenarios:
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab_sub, regional_mab=regional_mab,
            scenario_name=sc_name, disable_economic_presolve=False,
        )
        model = milp.model
        model.update()
        for (u, t), val in ev_stk.items():
            v = model.getVarByName(f'z[{u},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, s, t), val in ev_harv.items():
            v = model.getVarByName(f'h[{u},{s},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, t), val in ev_hexist.items():
            v = model.getVarByName(f'h_exist[{u},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, s), val in ev_q.items():
            v = model.getVarByName(f'q[{u},{s}]')
            if v is not None:
                v.LB = val; v.UB = val
        model.Params.OutputFlag = 0
        model.Params.MIPGap = mip_gap
        model.update()
        model.optimize()
        if model.SolCount > 0:
            n_feas += 1
            feas_prob += prob
            eev_obj += prob * model.ObjVal

    eev_normalized = eev_obj / feas_prob if feas_prob > 0 else 0.0
    return eev_normalized, time.time() - t0, n_feas


In [5]:
# Main experiment: {2, 3, 4} stages × {EEV, SP, WS}  —  60 cages, mip_gap=2%

MIP_GAP = 0.02
results = []

In [6]:
# 2-stage tree (9 scenarios)
n_stages = 2
n_sc = 3 ** n_stages
print(f'\n{"="*70}')
print(f'  {n_stages}-stage tree  ({n_sc} scenarios)  —  60 cages')
print(f'{"="*70}')

print(f'\n--- EEV ({n_stages}-stage) ---')
eev_obj, eev_time, eev_feas = run_eev_for(
    units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
)
print(f'  EEV = {eev_obj/1e6:.0f} MNOK  |  {eev_time:.0f}s  |  {eev_feas}/{n_sc} feasible')

print(f'\n--- SP ({n_stages}-stage) ---')
ALD = STAGE_CFG[n_stages]['ald_class']
ald = ALD(
    units_df=units_df_60, loc_mab=loc_mab_60, regional_mab=regional_mab,
    T=T, mip_gap=MIP_GAP,
    temps_bad=temps_bad_12, temps_normal=temps_normal_12, temps_good=temps_good_12,
)
ald.build()
ald.solve()
sp_obj  = ald.eval_obj
sp_time = ald.total_time
print(f'  SP  = {sp_obj/1e6:.0f} MNOK  |  {sp_time:.0f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')
del ald

print(f'\n--- WS ({n_stages}-stage) ---')
ws_obj, ws_time, ws_feas = run_ws_for(
    units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
)
print(f'  WS  = {ws_obj/1e6:.0f} MNOK  |  {ws_time:.0f}s  |  {ws_feas}/{n_sc} feasible')

vss_sp  = (sp_obj - eev_obj) / 1e6
evpi_sp = (ws_obj - sp_obj)  / 1e6
print(f'  VSS_SP={vss_sp:.0f} MNOK   EVPI_SP={evpi_sp:.0f} MNOK')

results.append({
    'stages':         n_stages,
    'scenarios':      n_sc,
    'EEV feas':       f'{eev_feas}/{n_sc}',
    'EEV [MNOK]':     round(eev_obj / 1e6, 0),
    'SP [MNOK]':      round(sp_obj  / 1e6, 0),
    'WS [MNOK]':      round(ws_obj  / 1e6, 0),
    'VSS_SP [MNOK]':  round(vss_sp,  0),
    'EVPI_SP [MNOK]': round(evpi_sp, 0),
    'EEV time [s]':   round(eev_time,  0),
    'SP time [s]':    round(sp_time,   0),
    'WS time [s]':    round(ws_time,   0),
    'MIP gap':        MIP_GAP,
})


  2-stage tree  (9 scenarios)  —  60 cages

--- EEV (2-stage) ---
Set parameter Username
Set parameter LicenseID to value 2786519
Academic license - for non-commercial use only - expires 2027-03-03
  EEV = 2071 MNOK  |  76s  |  6/9 feasible

--- SP (2-stage) ---
Built 9 scenarios
NA variables: 100140 total (100140 binary)
Variable cache built (tree-aware)


Step 0 (initial solve): 100%|██████████| 9/9 [01:38<00:00, 10.94s/sc]


Step 0 done — E[obj]: 3,821,732,051
Auto-calibrated rho_bin=7.118e+06, rho_max_bin=3.822e+10

Step 0 | Obj: 3821732050.89 | MaxDev: 8.89e-01 | AvgBin: 0.0021 | Fixed: 0
Starting PH iterations (max 1000)
  [Iter 1] heartbeat: 0/9 done, 9 running, oldest=5.0s | pending: 00,01,02,03,04,05,06,07,08
  [Iter 1] heartbeat: 0/9 done, 9 running, oldest=10.1s | pending: 00,01,02,03,04,05,06,07,08
  [Iter 1 sc=1] solved in 10.1s
  [Iter 1 sc=2] solved in 11.3s
  [Iter 1 sc=0] solved in 11.4s
  [Iter 1] progress: 3/9 done, 6 running
  [Iter 1 sc=6] solved in 12.2s
  [Iter 1 sc=7] solved in 13.9s
  [Iter 1 sc=5] solved in 14.7s
  [Iter 1] progress: 6/9 done, 3 running
  [Iter 1] heartbeat: 6/9 done, 3 running, oldest=15.9s | pending: 03,04,08
  [Iter 1 sc=4] solved in 16.7s
  [Iter 1 sc=3] solved in 17.8s
  [Iter 1] heartbeat: 8/9 done, 1 running, oldest=21.0s | pending: 08
  [Iter 1] heartbeat: 8/9 done, 1 running, oldest=26.0s | pending: 08
  [Iter 1 sc=8] solved in 16.2s
  [Iter 1] progress: 9/9

In [7]:
# 3-stage tree (27 scenarios)
n_stages = 3
n_sc = 3 ** n_stages
print(f'\n{"="*70}')
print(f'  {n_stages}-stage tree  ({n_sc} scenarios)  —  60 cages')
print(f'{"="*70}')

print(f'\n--- EEV ({n_stages}-stage) ---')
eev_obj, eev_time, eev_feas = run_eev_for(
    units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
)
print(f'  EEV = {eev_obj/1e6:.0f} MNOK  |  {eev_time:.0f}s  |  {eev_feas}/{n_sc} feasible')

print(f'\n--- SP ({n_stages}-stage) ---')
ALD = STAGE_CFG[n_stages]['ald_class']
ald = ALD(
    units_df=units_df_60, loc_mab=loc_mab_60, regional_mab=regional_mab,
    T=T, mip_gap=MIP_GAP,
    temps_bad=temps_bad_12, temps_normal=temps_normal_12, temps_good=temps_good_12,
)
ald.build()
ald.solve()
sp_obj  = ald.eval_obj
sp_time = ald.total_time
print(f'  SP  = {sp_obj/1e6:.0f} MNOK  |  {sp_time:.0f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')
del ald

print(f'\n--- WS ({n_stages}-stage) ---')
ws_obj, ws_time, ws_feas = run_ws_for(
    units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
)
print(f'  WS  = {ws_obj/1e6:.0f} MNOK  |  {ws_time:.0f}s  |  {ws_feas}/{n_sc} feasible')

vss_sp  = (sp_obj - eev_obj) / 1e6
evpi_sp = (ws_obj - sp_obj)  / 1e6
print(f'  VSS_SP={vss_sp:.0f} MNOK   EVPI_SP={evpi_sp:.0f} MNOK')

results.append({
    'stages':         n_stages,
    'scenarios':      n_sc,
    'EEV feas':       f'{eev_feas}/{n_sc}',
    'EEV [MNOK]':     round(eev_obj / 1e6, 0),
    'SP [MNOK]':      round(sp_obj  / 1e6, 0),
    'WS [MNOK]':      round(ws_obj  / 1e6, 0),
    'VSS_SP [MNOK]':  round(vss_sp,  0),
    'EVPI_SP [MNOK]': round(evpi_sp, 0),
    'EEV time [s]':   round(eev_time,  0),
    'SP time [s]':    round(sp_time,   0),
    'WS time [s]':    round(ws_time,   0),
    'MIP gap':        MIP_GAP,
})


  3-stage tree  (27 scenarios)  —  60 cages

--- EEV (3-stage) ---
  EEV = 2711 MNOK  |  157s  |  18/27 feasible

--- SP (3-stage) ---
Built 27 scenarios
NA variables: 231840 total (231840 binary)
Variable cache built (tree-aware)


Step 0 (initial solve): 100%|██████████| 27/27 [05:54<00:00, 13.11s/sc]


Step 0 done — E[obj]: 4,038,588,035
Auto-calibrated rho_bin=6.271e+06, rho_max_bin=4.039e+10

Step 0 | Obj: 4038588034.51 | MaxDev: 9.63e-01 | AvgBin: 0.0011 | Fixed: 0
Starting PH iterations (max 1000)
  [Iter 1] heartbeat: 0/27 done, 27 running, oldest=5.1s | pending: 00,01,02,03,04,05,06,07,08,09,10,11,...
  [Iter 1] heartbeat: 0/27 done, 27 running, oldest=10.1s | pending: 00,01,02,03,04,05,06,07,08,09,10,11,...
  [Iter 1] heartbeat: 0/27 done, 27 running, oldest=15.2s | pending: 00,01,02,03,04,05,06,07,08,09,10,11,...
  [Iter 1] no completions for 15.2s; 27 still running | slowest: sc=00:15.2s, sc=10:15.2s, sc=09:15.2s, sc=03:15.2s, sc=08:15.2s
  [Iter 1 sc=2] solved in 19.4s
  [Iter 1] heartbeat: 1/27 done, 26 running, oldest=20.7s | pending: 00,01,03,04,05,06,07,08,09,10,11,12,...
  [Iter 1 sc=5] solved in 21.3s
  [Iter 1] heartbeat: 2/27 done, 25 running, oldest=26.6s | pending: 00,01,03,04,06,07,08,09,10,11,12,13,...
  [Iter 1] heartbeat: 2/27 done, 25 running, oldest=31.7s | 

In [8]:
# 4-stage tree (81 scenarios)
n_stages = 4
n_sc = 3 ** n_stages
print(f'\n{"="*70}')
print(f'  {n_stages}-stage tree  ({n_sc} scenarios)  —  60 cages')
print(f'{"="*70}')

print(f'\n--- EEV ({n_stages}-stage) ---')
eev_obj, eev_time, eev_feas = run_eev_for(
    units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
)
print(f'  EEV = {eev_obj/1e6:.0f} MNOK  |  {eev_time:.0f}s  |  {eev_feas}/{n_sc} feasible')

print(f'\n--- SP ({n_stages}-stage) ---')
ALD = STAGE_CFG[n_stages]['ald_class']
ald = ALD(
    units_df=units_df_60, loc_mab=loc_mab_60, regional_mab=regional_mab,
    T=T, mip_gap=MIP_GAP,
    temps_bad=temps_bad_12, temps_normal=temps_normal_12, temps_good=temps_good_12,
)
ald.build()
ald.solve()
sp_obj  = ald.eval_obj
sp_time = ald.total_time
print(f'  SP  = {sp_obj/1e6:.0f} MNOK  |  {sp_time:.0f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')
del ald

print(f'\n--- WS ({n_stages}-stage) ---')
ws_obj, ws_time, ws_feas = run_ws_for(
    units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
)
print(f'  WS  = {ws_obj/1e6:.0f} MNOK  |  {ws_time:.0f}s  |  {ws_feas}/{n_sc} feasible')

vss_sp  = (sp_obj - eev_obj) / 1e6
evpi_sp = (ws_obj - sp_obj)  / 1e6
print(f'  VSS_SP={vss_sp:.0f} MNOK   EVPI_SP={evpi_sp:.0f} MNOK')

results.append({
    'stages':         n_stages,
    'scenarios':      n_sc,
    'EEV feas':       f'{eev_feas}/{n_sc}',
    'EEV [MNOK]':     round(eev_obj / 1e6, 0),
    'SP [MNOK]':      round(sp_obj  / 1e6, 0),
    'WS [MNOK]':      round(ws_obj  / 1e6, 0),
    'VSS_SP [MNOK]':  round(vss_sp,  0),
    'EVPI_SP [MNOK]': round(evpi_sp, 0),
    'EEV time [s]':   round(eev_time,  0),
    'SP time [s]':    round(sp_time,   0),
    'WS time [s]':    round(ws_time,   0),
    'MIP gap':        MIP_GAP,
})


  4-stage tree  (81 scenarios)  —  60 cages

--- EEV (4-stage) ---
  EEV = 2384 MNOK  |  339s  |  54/81 feasible

--- SP (4-stage) ---
Built 81 scenarios
NA variables: 532680 total (532680 binary)
Variable cache built (tree-aware)


Step 0 (initial solve): 100%|██████████| 81/81 [18:20<00:00, 13.58s/sc]


Step 0 done — E[obj]: 3,875,047,987
Auto-calibrated rho_bin=6.212e+06, rho_max_bin=3.875e+10

Step 0 | Obj: 3875047986.67 | MaxDev: 9.88e-01 | AvgBin: 0.0005 | Fixed: 0
Starting PH iterations (max 1000)
  [Iter 1] heartbeat: 0/81 done, 81 running, oldest=5.0s | pending: 00,01,02,03,04,05,06,07,08,09,10,11,...
  [Iter 1] heartbeat: 0/81 done, 81 running, oldest=10.1s | pending: 00,01,02,03,04,05,06,07,08,09,10,11,...
  [Iter 1] heartbeat: 0/81 done, 81 running, oldest=15.1s | pending: 00,01,02,03,04,05,06,07,08,09,10,11,...
  [Iter 1] no completions for 15.1s; 81 still running | slowest: sc=00:15.1s, sc=07:15.1s, sc=09:15.1s, sc=06:15.1s, sc=04:15.1s
  [Iter 1] heartbeat: 0/81 done, 81 running, oldest=20.2s | pending: 00,01,02,03,04,05,06,07,08,09,10,11,...
  [Iter 1 sc=7] solved in 24.9s
  [Iter 1] heartbeat: 1/81 done, 80 running, oldest=25.2s | pending: 00,01,02,03,04,05,06,08,09,10,11,12,...
  [Iter 1] heartbeat: 1/81 done, 80 running, oldest=30.3s | pending: 00,01,02,03,04,05,06,08

In [9]:
# Results table (Table 8)

df_results = pd.DataFrame(results).set_index('stages')
display(df_results)


,scenarios,EEV feas,EEV [MNOK],SP [MNOK],WS [MNOK],VSS_SP [MNOK],EVPI_SP [MNOK],EEV time [s],SP time [s],WS time [s],MIP gap
stages,,,,,,,,,,,
2,9,6/9,2071.0,3088.0,3824.0,1016.0,737.0,76.0,228.0,195.0,0.02
3,27,18/27,2711.0,3299.0,4042.0,588.0,743.0,157.0,1139.0,917.0,0.02
4,81,54/81,2384.0,3237.0,3876.0,853.0,639.0,339.0,3799.0,2190.0,0.02
